# Lab 1. Concordance and frequency

Reproduces Lab 1 on Austen's *Persuasion* and the essay corpora.

In [1]:
import json, sys, statistics
from collections import Counter
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, "../analysis")
from features import *

DATA = Path("../web/public/data")
essays = json.loads((DATA / "essays.json").read_text())["essays"]
wordlist = set(json.loads((DATA / "wordlist.json").read_text()))
markers = json.loads((DATA / "markers.json").read_text())
pd.set_option("display.precision", 3)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
TEAL, OCHRE, INK = "#008f86", "#b07d12", "#1c2b3a"


In [2]:
persuasion = (DATA / "persuasion.txt").read_text()
ptoks = tokens(persuasion)
print(len(ptoks), "words")

def kwic(text, node, span=60):
    import re
    flat = re.sub(r"\s+", " ", text)
    for m in re.finditer(rf"\b{re.escape(node)}\b", flat, flags=re.I):
        yield flat[max(0, m.start() - span):m.start()].rjust(span), m.group(0), flat[m.end():m.end() + span]

for left, node, right in list(kwic(persuasion, "persuaded"))[:15]:
    print(f"{left} [{node}] {right}")

83616 words
 reality of Sir Walter's retrenching, and who was perfectly  [persuaded]  that nothing would be done without a change of abode. "Sinc
ess of manner, be continually advising her in vain. She was  [persuaded]  to believe the engagement a wrong thing: indiscreet, improp
immediate wretchedness, such uncertain future good. She was  [persuaded]  that under every disadvantage of disapprobation at home, an
in you very much, I am very fond of a long walk;" Anne felt  [persuaded] , by the looks of the two girls, that it was precisely what 
any person I may say? No, I have no idea of being so easily  [persuaded] . When I have made up my mind, I have made it; and Henrietta
kish enough to please Lady Russell, and that therefore, she  [persuaded]  Anne to refuse him." The sounds were retreating, and Anne d
ar how soon we came to an understanding, she would never be  [persuaded]  that we could be happy together. I had known you by charact
enrietta at first declared the same. She, however,

In [3]:
lines = list(kwic(persuasion, "persuaded"))
print(Counter((r.split() or [""])[0].lower().strip(",.;") for _, _, r in lines).most_common(8))

[('that', 4), ('to', 4), ('', 2), ('by', 2), ('anne', 1), ('them', 1)]


## Frequency, with and without function words

In [4]:
STOP = set("""the a an and or but of to in on at for with by from as is are was were be been being it its this that these those
he she they them his her their i you we my your our me him us not no so if then than there here what which who whom when where why
how all any some such into out up down over under again very can could would should will shall may might must do does did have has had am""".split())
f = Counter(ptoks)
pd.DataFrame({"all words": [w for w, _ in f.most_common(20)],
              "content words": [w for w, _ in ((w, c) for w, c in f.most_common() if w not in STOP)][:20]})

,all words,content words
0,the,anne
1,to,captain
2,and,mrs
3,of,more
4,a,mr
5,in,elliot
6,was,one
7,her,only
8,had,lady
9,she,much


In [5]:
for src in ["human", "llm"]:
    t = [w for e in essays if e["source"] == src for w in tokens(e["text"])]
    print(src, "'because' per 1,000 words:", round(1000 * t.count("because") / len(t), 2))

human 'because' per 1,000 words: 10.97
llm 'because' per 1,000 words: 2.73
